In [1]:
from matplotlib import pyplot as plt
import numpy as np
from scipy.special import jv, kv
from scipy.special import j1 # Bessel function of the first kind of order 1, for the diffraction pattern calculation
from scipy.optimize import root_scalar
from scipy.constants import c, epsilon_0, mu_0, pi
import scipy.integrate as integrate

In [2]:
# jn_zeros is a function that returns the zeros of the Bessel function of the first kind, 
# which are needed to find the roots for the LP01 mode in a step-index fibre
from scipy.special import jn_zeros  # NEW: needed to get the first zero of J0

In [3]:
import sys
from pathlib import Path

src_path = Path("/Users/tliu/oh_suppression/codes/src")

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from oh_suppression.coupling import *

In [4]:
from oh_suppression.point_source_coupling import *
from oh_suppression.flat_background_coupling import *
from oh_suppression.extended_source_coupling import *

In [5]:
from matplotlib import pyplot as plt
import numpy as np
from scipy import special #import jv, kv
# from scipy.special import j1 # Bessel function of the first kind of order 1, for the diffraction pattern calculation
from scipy.optimize import root_scalar
from scipy.constants import c, epsilon_0, mu_0, pi
import scipy.integrate as integrate
# jn_zeros is a function that returns the zeros of the Bessel function of the first kind, 
# which are needed to find the roots for the LP01 mode in a step-index fibre
from scipy.special import jn_zeros  # NEW: needed to get the first zero of J0
from scipy.optimize import brentq
from scipy.optimize import fminbound
from dataclasses import dataclass
import pandas as pd
from joblib import Parallel, delayed
import os

In [6]:
import astropy.units as u
from astropy.modeling.models import Moffat1D
# Source: https://docs.astropy.org/en/stable/api/astropy.modeling.functional_models.Moffat1D.html
from astropy.modeling.functional_models import Moffat2D
# Source: https://docs.astropy.org/en/stable/api/astropy.modeling.functional_models.Moffat2D.html


In [7]:
def Moffat_1d_profile(amplitude, x_0, gamma, alpha): 
    """
    Returns a 1D Moffat profile function.
    
    Parameters:
    amplitude : float
        The peak value of the profile.
    x_0 : float
        The position of the peak.
    gamma : float
        The scale parameter, related to the FWHM.
    alpha : float
        The shape parameter, controlling the wings of the profile.
    
    Returns:
    function
        A function that computes the Moffat profile at given x values.
    """
    def profile(x):
        return amplitude * (1 + ((x - x_0) / gamma) ** 2) ** (-alpha)
    
    return profile

In [ ]:
def Monte_Carlo_point_source_coupling(
        delta_x, delta_y, 
        eta_psf_interp, 
        acceptance_radius=None, 
): 
    """
    Perform a Monte Carlo simulation to estimate the coupling efficiency of a point source in 
    the presence of tip-tilt jitter. 

    For each random jitter position, the radial displacement of the point source is calcualted. 

    We should provide the function with an acceptance radius as point source outside of 
    the acceptance solid angle of sky will not be coupled into the fibre. 

    """
